# SRM Melhoria de Processo

## Bibliotecas Usadas

In [29]:
import pandas as pd
import time
import numpy as np

In [30]:
 # Parametros

periodo = 3
ano = 2026

## Montagem da Base N13P

In [31]:
#-#-#-# Inicio da contagem do tempo de execução #-#-#-#
inicio_total = time.time()
inicio_ciclo = time.time()


# Ciclo

colunas_base = ['Regional', 'GP', 'Gerente', 'Rede', 'COD_CLIENTE', 'Company Code','CD',
 'NOME_CLIENTE', 'UF', 'Região', 'EAN', 'SKU', 'Desc. SKU', 'Classificação', 'Subbrand','Marca']

colunas_periodos = [f'P{i:02d}-{ano}' for i in range(periodo, 14)]

colunas = colunas_base + colunas_periodos

tipos_colunas ={'EAN': str, 
                'COD_CLIENTE': str,
                'Company Code': str, 
                'CD': str, 
                'SKU': str}

df_ciclo_n13 = pd.read_excel(f'../data/Arquivos/Ciclo_P{periodo:02d} N13P {ano} - envio.xlsx', header=3, usecols=colunas,
                             dtype=tipos_colunas,
                             engine='calamine')


# Clientes

tipos_colunas_clientes = {'COD_CLIENTE': str, 'NOME_CLIENTE': str, 'COD REDE': str, 'COD SUBREDE': str, 'COND. PAG': str}

df_clientes = pd.read_excel('../data/Arquivos/BASE CLIENTES.xlsx', 
                            dtype=tipos_colunas_clientes,
                            engine='calamine')


# Produtos

tipos_colunas ={'EAN': str, 
                'SKU': str,
                'Ton/CDA': float, 
                'Unid/\nCX	': float, 
                'Hierarquia': str,
                'NCM': str,
                'kg/Un': float,
                'H05': str,
                'LSV': float}

df_produtos = pd.read_excel('../data/Arquivos/BASE PRODUTOS.xlsx', 
                            dtype=tipos_colunas,
                            engine='calamine')

df_produtos = df_produtos.rename(columns={'Unid/\nCX': 'Unid/CDA', 'kg/Un': 'kg/UN'})

#-#-#-# Colunas especificas #-#-#-#

#  UF Origem

dicionario_uf = {
    'BR01': 'SP',
    'BR03': 'PE',
    'BR30': 'SP',
    'BR31': 'MG',
}

df_ciclo_n13['UF ORIGEM'] = df_ciclo_n13['CD'].map(dicionario_uf)

df_ciclo_n13[['CD', 'UF ORIGEM']].head(10)

# COD GP

dicionario_gp = {
    'ATACADO CASH & CARRY': 'AG',
    'GPA': 'AH',
    "SAM'S CLUB": 'AM',
    'GROCERY': 'AL',
    'ASSAI': 'AX',
    'Atacadão': 'TA',
    'DIST. MISTO': 'BI',
    'ESPECIALISTA DIRETO': 'AD',
    'DIA %': 'AF',
    'DIST. ALIMENTAR': 'AI',
    'DIST. ESPECIALISTA': 'AJ',
    'CENCOSUD': 'BJ',
    'CARREFOUR': 'AE',
    'ECOMMERCE': 'BO',
    'KA ESPECIALISTA': 'AQ',
    'PETZ': 'BL',
    'ATACADOS': 'AB',
    'MARTINS': 'BK',
    'COBASI': 'BM',
    'ATACADOS ESPECIAIS': 'BT',
    'CONVENIENCIAS': 'BP'
}

df_ciclo_n13['CÓD GP'] = df_ciclo_n13['GP'].map(dicionario_gp)

df_ciclo_n13['GP'] = df_ciclo_n13['GP'].str.strip()

# EAN Espelho

search_ean = df_produtos.set_index('EAN', drop=False)['EAN'].to_dict()
df_ciclo_n13['EAN Espelho'] = df_ciclo_n13['EAN'].map(search_ean)

search_desc = df_produtos.set_index('Descrição', drop=False)['EAN'].to_dict()
secondary_search = df_ciclo_n13['Desc. SKU'].map(search_desc)
df_ciclo_n13['EAN Espelho'] = df_ciclo_n13['EAN Espelho'].fillna(secondary_search)

# SKU

colunas_produtos = ['EAN', 'SKU', 'Family Price', 'Hierarquia', 'Class.', 'NCM', 
                    'Origem', 'kg/UN', 'Ton/CDA', 'Unid/CDA', 'LSV']

df_prod_exato = df_produtos[colunas_produtos].drop_duplicates(subset=['EAN', 'SKU'], keep='first')
df_prod_resgate = df_produtos[colunas_produtos].drop(columns=['SKU']).drop_duplicates(subset=['EAN'], keep='first')

df_ciclo_n13 = pd.merge(df_ciclo_n13,
                        df_prod_exato,
                        left_on=['EAN Espelho', 'SKU'],
                        right_on=['EAN', 'SKU'],
                        how='left')

df_ciclo_n13 = df_ciclo_n13.drop(columns=['EAN_y'], errors='ignore')
df_ciclo_n13 = df_ciclo_n13.rename(columns={'EAN_x': 'EAN'})

df_ciclo_n13 = pd.merge(df_ciclo_n13,
                        df_prod_resgate,
                        left_on='EAN Espelho',
                        right_on='EAN',
                        how='left',
                        suffixes=('', '_resgate'))

colunas_preencher = ['Family Price', 'Hierarquia', 'Class.', 'NCM', 'Origem', 'kg/UN', 'Ton/CDA', 'Unid/CDA', 'LSV']

for col in colunas_preencher:
    df_ciclo_n13[col] = df_ciclo_n13[col].fillna(df_ciclo_n13[f'{col}_resgate'])

colunas_lixo = [f'{col}_resgate' for col in colunas_preencher] + ['EAN_resgate']
df_ciclo_n13 = df_ciclo_n13.drop(columns=colunas_lixo, errors='ignore')

# Codigo Subrede

df_clientes_limpo = df_clientes[['COD_CLIENTE', 'COD REDE','COD SUBREDE', 'COND. PAG']].drop_duplicates(subset=['COD_CLIENTE'], keep='first')

df_ciclo_n13 = pd.merge(
    df_ciclo_n13,
    df_clientes_limpo, 
    on='COD_CLIENTE',
    how='left'
)

fim_ciclo = time.time()
tempo_ciclo = fim_ciclo - inicio_ciclo

In [32]:
# for col in df_ciclo_n13.columns:
#     print(f"{col}: {df_ciclo_n13[col].dtype}")

## Calculos

### ZP55

In [33]:
inicio_zp55 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp55 = pd.read_excel('../data/Arquivos/ZP55.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp55['Cadastro'] = df_zp55['Cadastro'].round(4)

# Busca

df_zp55['CHAVE'] = df_zp55['CHAVE'].astype(str).str.strip()
dic_zp55 = df_zp55.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['Cadastro'].to_dict()


chave_1_full = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip()
chave_1_10 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip().str[:10]

chave_2 = df_ciclo_n13['CD'].astype(str).str.strip() + '_' + df_ciclo_n13['UF'].astype(str).str.strip() + '_' + df_ciclo_n13['Origem'].astype(str).str.strip()
chave_3 = df_ciclo_n13['CD'].astype(str).str.strip() + '_' + df_ciclo_n13['UF'].astype(str).str.strip() + '_' + df_ciclo_n13['NCM'].astype(str).str.strip()
chave_4 = df_ciclo_n13['CD'].astype(str).str.strip() + '_' + df_ciclo_n13['UF'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip().str[:10]

df_ciclo_n13['CLIENTE'] = (chave_1_full.map(dic_zp55).fillna(chave_1_10.map(dic_zp55)) / 100)
df_ciclo_n13['CD + UF DESTINO + Importação'] = (chave_2.map(dic_zp55) / 100)
df_ciclo_n13['CD + UF DESTINO + NCM'] = (chave_3.map(dic_zp55) / 100)
df_ciclo_n13['CD + UF DESTINO + H05'] = (chave_4.map(dic_zp55) / 100)

df_ciclo_n13['ZP55'] = (df_ciclo_n13['CLIENTE']
                        .fillna(df_ciclo_n13['CD + UF DESTINO + Importação'])
                        .fillna(df_ciclo_n13['CD + UF DESTINO + NCM'])
                        .fillna(df_ciclo_n13['CD + UF DESTINO + H05'])
                        )

df_ciclo_n13['ZP55'] = df_ciclo_n13['ZP55'].round(4)

fim_zp55 = time.time()
tempo_zp55 = fim_zp55 - inicio_zp55

### ZP 54

In [34]:
inicio_zp54 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp54 = pd.read_excel('../data/Arquivos/ZP54.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp54['Cadastro'] = df_zp54['Cadastro'].round(4)

# Chaves

dic_zp54 = df_zp54.set_index('CHAVE')['Cadastro'].to_dict()

chave_1_12 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:12]
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:12]
chave_3 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['CÓD GP'].astype(str) + ' ' + df_ciclo_n13['UF'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:12]
chave_4 = df_ciclo_n13['Company Code'].astype(str) + '_' +df_ciclo_n13['CÓD GP'].astype(str) + ' ' + df_ciclo_n13['UF'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:10]

# Busca

df_ciclo_n13['1. CLIENTE'] = (chave_1_12.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. REDE'] = (chave_2.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. GP UF HIER 6'] = (chave_3.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. GP UF HIER 5'] = (chave_4.map(dic_zp54) / 100).round(4)

df_ciclo_n13['ZP54'] = (df_ciclo_n13['1. CLIENTE']
                        .fillna(df_ciclo_n13['1. REDE'])
                        .fillna(df_ciclo_n13['1. GP UF HIER 6'])
                        .fillna(df_ciclo_n13['1. GP UF HIER 5'])
                        )

df_ciclo_n13['ZP54'] = df_ciclo_n13['ZP54'].round(4)

fim_zp54 = time.time()
tempo_zp54 = fim_zp54 - inicio_zp54

### GSVs

In [35]:
inicio_gsv = time.time()

# GSV/CDA

df_ciclo_n13['GSV/CDA'] = df_ciclo_n13['LSV'] * (1 + df_ciclo_n13['ZP55']) * (1 + df_ciclo_n13['ZP54'])

df_ciclo_n13['GSV/CDA'] = df_ciclo_n13['GSV/CDA'].round(4)

# GSV/TON

coluna_BA = df_ciclo_n13['GSV/CDA'] 
coluna_AA = df_ciclo_n13['kg/UN'] 
coluna_AO = df_ciclo_n13['Unid/CDA'] 

denominador = coluna_AA * coluna_AO

df_ciclo_n13['GSV/TON'] = np.where(
    (denominador == 0) | (denominador.isna()),
    np.nan,                                   
    (coluna_BA / denominador) * 1000           
)

df_ciclo_n13['GSV/TON'] = df_ciclo_n13['GSV/TON'].round(4)


fim_gsv = time.time()
tempo_gsv = fim_gsv - inicio_gsv

### Projeções

In [36]:
inicio_projecao = time.time()

for p in colunas_periodos:
    nome_coluna_projecao = f'GSV R$ {p} - {ano}'

    df_ciclo_n13[nome_coluna_projecao] = df_ciclo_n13[p] * df_ciclo_n13['GSV/TON']

fim_projecao = time.time()
tempo_projecao = fim_projecao - inicio_projecao

### ZP53

In [37]:
inicio_zp53 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float, 'P\'ANO_FIM' : str}

colunas = ['CHAVE', 'Cadastro', 'P\'ANO_FIM']

df_zp53 = pd.read_excel('../data/Arquivos/ZP53.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp53['Cadastro'] = df_zp53['Cadastro'].round(2)

df_zp53['CHAVE'] = df_zp53['CHAVE'].astype(str).str.strip()

# Chaves

dic_zp531 = df_zp53.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['Cadastro'].to_dict()
dic_zp532 = df_zp53.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['P\'ANO_FIM'].to_dict()

chave_1 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip()
chave_2 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['COD SUBREDE'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip()
chave_3 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['CÓD GP'].astype(str).str.strip() + ' ' + df_ciclo_n13['UF'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip()
chave_4 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['CÓD GP'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip().str[:10] # <-- CORREÇÃO AQUI!

# Busca
df_ciclo_n13['1. EMISSOR'] = chave_1.map(dic_zp531) / 100
df_ciclo_n13['1. REDE'] = chave_2.map(dic_zp531) / 100
df_ciclo_n13['1. GP UF'] = chave_3.map(dic_zp531) / 100
df_ciclo_n13['1. GP'] = chave_4.map(dic_zp531) / 100

df_ciclo_n13['2. EMISSOR'] = chave_1.map(dic_zp532)
df_ciclo_n13['2. REDE'] = chave_2.map(dic_zp532)
df_ciclo_n13['2. GP UF'] = chave_3.map(dic_zp532)
df_ciclo_n13['2. GP'] = chave_4.map(dic_zp532)

df_ciclo_n13['ZP53'] = (df_ciclo_n13['1. EMISSOR']
                        .fillna(df_ciclo_n13['1. REDE'])
                        .fillna(df_ciclo_n13['1. GP UF'])
                        .fillna(df_ciclo_n13['1. GP'])
                        .fillna(0) 
                    )

df_ciclo_n13['ZP53'] = df_ciclo_n13['ZP53'].round(4)

fim_zp53 = time.time()
tempo_zp53 = fim_zp53 - inicio_zp53

### ZP52

In [38]:
inicio_zp52 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp52 = pd.read_excel('../data/Arquivos/ZP52.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp52['Cadastro'] = df_zp52['Cadastro'].round(2)

df_zp52['CHAVE'] = df_zp52['CHAVE'].astype(str).str.strip()

# Chaves

dic_zp52 = df_zp52.set_index('CHAVE')['Cadastro'].to_dict()

chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:8]
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:2]

# Busca
df_ciclo_n13['H04'] = (chave_1.map(dic_zp52) / 100).round(4)
df_ciclo_n13['H01'] = (chave_2.map(dic_zp52) / 100).round(4)

df_ciclo_n13['ZP52'] = (df_ciclo_n13['H04']
                        .fillna(df_ciclo_n13['H01'])
                        .fillna(0)
                        )
                        
df_ciclo_n13['ZP52'] = df_ciclo_n13['ZP52'].round(4)

fim_zp52 = time.time()
tempo_zp52 = fim_zp52 - inicio_zp52

### ZP73

In [39]:
inicio_zp73 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp73 = pd.read_excel('../data/Arquivos/ZP73.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp73['Cadastro'] = df_zp73['Cadastro'].round(2)

df_zp73['CHAVE'] = df_zp73['CHAVE'].astype(str).str.strip()

# Chaves

dic_zp73 = df_zp73.set_index('CHAVE')['Cadastro'].to_dict()

chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str)
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str)

df_ciclo_n13['ZP73'] = ((chave_1.map(dic_zp73) / 100).round(4)
                        .fillna((chave_2.map(dic_zp73) / 100).round(4))
                        .fillna(0)
                        )
                        
df_ciclo_n13['ZP73'] = df_ciclo_n13['ZP73'].round(4)

fim_zp73 = time.time()
tempo_zp73 = fim_zp73 - inicio_zp73

### ZP70

In [40]:
inicio_zp70 = time.time()

# Importação

tipos_colunas = {'CONDICAO DE PAGAMENTO' : str, 'Desconto': float}

colunas = ['CONDICAO DE PAGAMENTO', 'Desconto']

df_zp70 = pd.read_excel('../data/Arquivos/ZP70.xlsx', 
                        header=0,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp70['Desconto'] = df_zp70['Desconto'].round(4)

# Busca

dic_zp70 = df_zp70.set_index('CONDICAO DE PAGAMENTO')['Desconto'].to_dict()

chave_1 = df_ciclo_n13['COND. PAG'].astype(str)

df_ciclo_n13['ZP70'] = ((chave_1.map(dic_zp70)).round(4)
                        .fillna(0)
                        )
                        
df_ciclo_n13['ZP70'] = df_ciclo_n13['ZP70'].round(4)

fim_zp70 = time.time()
tempo_zp70 = fim_zp70 - inicio_zp70

### ZP39

In [41]:
inicio_zp39 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float, 'P\'ANO_FIM': str}

colunas = ['CHAVE', 'Cadastro', 'P\'ANO_FIM']

df_zp39 = pd.read_excel('../data/Arquivos/ZP39.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp39['Cadastro'] = df_zp39['Cadastro'].round(4)

# Busca

dic_zp391 = df_zp39.set_index('CHAVE')['Cadastro'].to_dict()
dic_zp392 = df_zp39.set_index('CHAVE')['P\'ANO_FIM'].to_dict()

chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:10]

df_ciclo_n13['1. ZP39'] = ((chave_1.map(dic_zp391) / 100).round(4)
                        .fillna(0)
                        )

df_ciclo_n13['2. ZP39'] = chave_1.map(dic_zp392)

df_ciclo_n13['1. ZP39'] = df_ciclo_n13['1. ZP39'].round(4)

fim_zp39 = time.time()
tempo_zp39 = fim_zp39 - inicio_zp39

### NIV

In [42]:
inicio_niv = time.time()

# NIV/CDA

df_ciclo_n13['NIV/CDA'] = df_ciclo_n13['GSV/CDA'] * (1 + df_ciclo_n13['ZP53']) * (1 + df_ciclo_n13['ZP52']) * (1 + df_ciclo_n13['ZP73']) * (1 + df_ciclo_n13['ZP70']) * (1 + df_ciclo_n13['1. ZP39']) 

df_ciclo_n13['NIV/CDA'] = df_ciclo_n13['NIV/CDA'].round(4)

# NIV/TON

coluna_BA = df_ciclo_n13['NIV/CDA']
coluna_AA = df_ciclo_n13['kg/UN'] 
coluna_AO = df_ciclo_n13['Unid/CDA'] 

denominador = coluna_AA * coluna_AO

df_ciclo_n13['NIV/TON'] = np.where(
    (denominador == 0) | (denominador.isna()),
    np.nan,                                   
    (coluna_BA / denominador) * 1000           
)

df_ciclo_n13['NIV/TON'] = df_ciclo_n13['NIV/TON'].round(4)


fim_niv = time.time()
tempo_niv = fim_niv - inicio_niv

### Importação dos impostos

In [43]:
pis = 0.0165
cofins = 0.076

tipos_colunas = {
    'CHAVE_1' : str,
    'CHAVE_2' : str,
    'Aliq_COFINS': float,
    'Aliq_PIS': float,
                }

colunas = ['CHAVE_1', 'CHAVE_2', 'Aliq_COFINS', 'Aliq_PIS']

df_impostos = pd.read_excel('../data/Arquivos/SPIM - MacGyver v102.xlsm', 
                        header=2,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine',
                        sheet_name='BD_Impostos_Oficial')

df_impostos['Aliq_COFINS'] = df_impostos['Aliq_COFINS'].round(4)
df_impostos['Aliq_PIS'] = df_impostos['Aliq_PIS'].round(4)

df_impostos['CHAVE_1'] = df_impostos['CHAVE_1'].astype(str).str.strip()
df_impostos['CHAVE_2'] = df_impostos['CHAVE_2'].astype(str).str.strip()

# df_impostos.head(50)

### PIS/COFINS

In [44]:
for col in df_ciclo_n13.columns:
    print(f"{col}: {df_ciclo_n13[col].name}")

Regional: Regional
GP: GP
Gerente: Gerente
Rede: Rede
COD_CLIENTE: COD_CLIENTE
Company Code: Company Code
CD: CD
NOME_CLIENTE: NOME_CLIENTE
UF: UF
Região: Região
EAN: EAN
SKU: SKU
Desc. SKU: Desc. SKU
Classificação: Classificação
Subbrand: Subbrand
Marca: Marca
P03-2026: P03-2026
P04-2026: P04-2026
P05-2026: P05-2026
P06-2026: P06-2026
P07-2026: P07-2026
P08-2026: P08-2026
P09-2026: P09-2026
P10-2026: P10-2026
P11-2026: P11-2026
P12-2026: P12-2026
P13-2026: P13-2026
UF ORIGEM: UF ORIGEM
CÓD GP: CÓD GP
EAN Espelho: EAN Espelho
Family Price: Family Price
Hierarquia: Hierarquia
Class.: Class.
NCM: NCM
Origem: Origem
kg/UN: kg/UN
Ton/CDA: Ton/CDA
Unid/CDA: Unid/CDA
LSV: LSV
COD REDE: COD REDE
COD SUBREDE: COD SUBREDE
COND. PAG: COND. PAG
CLIENTE: CLIENTE
CD + UF DESTINO + Importação: CD + UF DESTINO + Importação
CD + UF DESTINO + NCM: CD + UF DESTINO + NCM
CD + UF DESTINO + H05: CD + UF DESTINO + H05
ZP55: ZP55
1. CLIENTE: 1. CLIENTE
1. REDE: 1. REDE
1. GP UF HIER 6: 1. GP UF HIER 6
1. GP 

In [51]:
dic_chave_1_impostos_cofins = df_impostos.drop_duplicates(subset=['CHAVE_1'], keep='first').set_index('CHAVE_1')["Aliq_COFINS"].to_dict()
dic_chave_1_impostos_pis = df_impostos.drop_duplicates(subset=['CHAVE_1'], keep='first').set_index('CHAVE_1')["Aliq_PIS"].to_dict()

dic_chave_2_impostos_cofins = df_impostos.drop_duplicates(subset=['CHAVE_2'], keep='first').set_index('CHAVE_2')["Aliq_COFINS"].to_dict()
dic_chave_2_impostos_pis = df_impostos.drop_duplicates(subset=['CHAVE_2'], keep='first').set_index('CHAVE_2')["Aliq_PIS"].to_dict()


chave_1 = df_ciclo_n13['Subbrand'].astype(str).str.strip() + df_ciclo_n13['UF ORIGEM'].astype(str).str.strip() + df_ciclo_n13['UF'].astype(str).str.strip()
chave_2 = df_ciclo_n13['UF ORIGEM'].astype(str).str.strip() + df_ciclo_n13['UF'].astype(str).str.strip()


df_ciclo_n13['Pis'] = (chave_1.map(dic_chave_1_impostos_pis).fillna(chave_2.map(dic_chave_2_impostos_pis)) / 100)


# df_ciclo_n13['Pis', 'Cofins'] = df_ciclo_n13['Pis', 'Cofins'].round(4)

# df_ciclo_n13['Pis', 'Cofins'] = df_ciclo_n13['Pis', 'Cofins'].fillna(0)

df_ciclo_n13[['Subbrand', 'UF ORIGEM', 'UF','Pis']].head(50)


,Subbrand,UF ORIGEM,UF,Pis
0,PED DRY,SP,SP,0.000165
1,PED DRY,SP,SP,0.000165
2,PED DRY,SP,SP,0.000165
3,PED DRY,SP,SP,0.000165
4,PED DRY,MG,PR,0.000165
5,PED DRY,MG,PR,0.000165
6,PED DRY,MG,PR,0.000165
7,PED DRY,MG,PR,0.000165
8,PED DRY,SP,SP,0.000165
9,PED DRY,SP,SP,0.000165


In [3]:
inicio_pis_cofins = time.time()

# Importação

df_ciclo_n13['BASE CALCULO PIS/COFINS'] = df_ciclo_n13['NIV/CDA'] / (1 - pis - cofins)

df_ciclo_n13['PIS/COFINS ABS'] = df_ciclo_n13['BASE CALCULO PIS/COFINS'] * (pis + cofins)

print(df_ciclo_n13[['NIV/CDA','BASE CALCULO PIS/COFINS', 'PIS/COFINS ABS']].head())

fim_pis_cofins = time.time()
tempo_pis_cofins = fim_pis_cofins - inicio_pis_cofins



NameError: name 'time' is not defined

## Exportação

In [ ]:
df_ciclo_n13.columns.to_list()

['Regional',
 'GP',
 'Gerente',
 'Rede',
 'COD_CLIENTE',
 'Company Code',
 'CD',
 'NOME_CLIENTE',
 'UF',
 'Região',
 'EAN',
 'SKU',
 'Desc. SKU',
 'Classificação',
 'Marca',
 'P03-2026',
 'P04-2026',
 'P05-2026',
 'P06-2026',
 'P07-2026',
 'P08-2026',
 'P09-2026',
 'P10-2026',
 'P11-2026',
 'P12-2026',
 'P13-2026',
 'UF ORIGEM',
 'CÓD GP',
 'EAN Espelho',
 'Family Price',
 'Hierarquia',
 'Class.',
 'NCM',
 'Origem',
 'kg/UN',
 'Ton/CDA',
 'Unid/CDA',
 'LSV',
 'COD REDE',
 'COD SUBREDE',
 'COND. PAG',
 'CLIENTE',
 'CD + UF DESTINO + Importação',
 'CD + UF DESTINO + NCM',
 'CD + UF DESTINO + H05',
 'ZP55',
 '1. CLIENTE',
 '1. REDE',
 '1. GP UF HIER 6',
 '1. GP UF HIER 5',
 'ZP54',
 'GSV/CDA',
 'GSV/TON',
 'GSV R$ P03-2026 - 2026',
 'GSV R$ P04-2026 - 2026',
 'GSV R$ P05-2026 - 2026',
 'GSV R$ P06-2026 - 2026',
 'GSV R$ P07-2026 - 2026',
 'GSV R$ P08-2026 - 2026',
 'GSV R$ P09-2026 - 2026',
 'GSV R$ P10-2026 - 2026',
 'GSV R$ P11-2026 - 2026',
 'GSV R$ P12-2026 - 2026',
 'GSV R$ P13-2026 

In [ ]:
%pip install xlsxwriter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# import time

inicio_export = time.time()

# cols_base = ['Regional',
#             'GP',
#             'Gerente',
#             'Rede',
#             'COD_CLIENTE',
#             'Company Code',
#             'CD',
#             'NOME_CLIENTE',
#             'UF',
#             'Região',
#             'EAN',
#             'SKU',
#             'Desc. SKU',
#             'Classificação',
#             'Marca',
#             'LSV'] 

# cols_produtos = ['UF ORIGEM',
#                 'CÓD GP',
#                 'Family Price',
#                 'Hierarquia',
#                 'Class.',
#                 'NCM',
#                 'Origem',
#                 'kg/UN',
#                 'Ton/CDA',
#                 'Unid/CDA',]

# col_ean = 'EAN Espelho'

# cols_zps = ['ZP55', 'ZP54', 'ZP52', 'ZP52', 'ZP53', 'ZP39', 'ZP70', 'ZP73']
# cols_financeiras = ['GSV/CDA', 'GSV/TON', 'NIV/CDA', 'NIV/TON']
# cols_chaves = ['CLIENTE',
#             'CD + UF DESTINO + Importação',
#             'CD + UF DESTINO + NCM',
#             'CD + UF DESTINO + H05',
#             '1. CLIENTE',
#             '1. REDE',
#             '1. GP UF HIER 6',
#             '1. GP UF HIER 5', 
#             '1. EMISSOR',
#             '1. GP UF',
#             '1. GP',
#             '2. EMISSOR',
#             '2. REDE',
#             '2. GP UF',
#             '2. GP',
#             'H04',
#             'H01',]


# writer = pd.ExcelWriter('N13_Final_Organizado.xlsx', engine='xlsxwriter')
# df_ciclo_n13.to_excel(writer, sheet_name='Valoracao', index=False)

# workbook  = writer.book
# worksheet = writer.sheets['Valoracao']

# # Estilos
# estilo_header = {'bold': True, 'border': 1, 'align': 'center', 'valign': 'vcenter'}

# fmt_padrao   = workbook.add_format({**estilo_header, 'bg_color': '#FFFFFF'}) 
# fmt_roxo     = workbook.add_format({**estilo_header, 'bg_color': "#7E306A", 'font_color': "#FFFFFF"})
# fmt_vermelho = workbook.add_format({**estilo_header, 'bg_color': "#DF3416", 'font_color': "#FFFFFF"})
# fmt_azul     = workbook.add_format({**estilo_header, 'bg_color': "#0753A5", 'font_color': "#FFFFFF"})
# fmt_cinza    = workbook.add_format({**estilo_header, 'bg_color': "#5A5A5A", 'font_color': "#FFFFFF"})
# fmt_agua     = workbook.add_format({**estilo_header, 'bg_color': "#00FFFF", 'font_color': "#000000"})
# fmt_verde     = workbook.add_format({**estilo_header, 'bg_color': "#06E92C", 'font_color': "#000000"})


# for col_num, col_nome in enumerate(df_ciclo_n13.columns):
    
#     # Escolhe o formato baseado na lista
#     if col_nome in cols_base:
#         formato = fmt_padrao
#     elif col_nome in cols_produtos:
#         formato = fmt_roxo
#     elif col_nome == col_ean:
#         formato = fmt_vermelho
#     elif col_nome in cols_zps:
#         formato = fmt_azul
#     elif col_nome in cols_financeiras:
#         formato = fmt_agua
#     elif col_nome in cols_chaves:
#         formato = fmt_cinza
#     elif col_nome in cols_financeiras or col_nome.startswith(('GSV R$', 'TON P')):
#         formato = fmt_verde
#     else:
#         formato = fmt_padrao

#     # Aplica a cor no cabeçalho (linha 0)
#     worksheet.write(0, col_num, col_nome, formato)
    
#     # Auto-ajuste de largura simples
#     largura = max(len(col_nome), 10) + 2
#     worksheet.set_column(col_num, col_num, largura)

# # Finaliza
# writer.close()

fim_export = time.time()
tempo_exportacao = fim_export - inicio_export

## Métricas

In [ ]:
fim_total = time.time()
tempo_total = fim_total - inicio_total


print('# # # Tempos de Execução # # #\n\n'
'Ciclo: {:.2f} segundos\n'
'ZP55: {:.2f} segundos\n'
'ZP54: {:.2f} segundos\n'
'GSV: {:.2f} segundos\n'
'Projeção: {:.2f} segundos\n'
'ZP53: {:.2f} segundos\n'
'ZP52: {:.2f} segundos\n'
'ZP73: {:.2f} segundos\n'
'ZP70: {:.2f} segundos\n'
'ZP39: {:.2f} segundos\n'
'NIV: {:.2f} segundos\n'
'Exportação: {:.2f} segundos\n'
'Total: {:.2f} segundos'
.format(tempo_ciclo, tempo_zp55, tempo_zp54, tempo_gsv, tempo_projecao, tempo_zp53, tempo_zp52, tempo_zp73, tempo_zp70, tempo_zp39, tempo_niv, tempo_exportacao, tempo_total))

print('')

num_linhas = len(df_ciclo_n13)
print(f'Número de linhas do df_ciclo_n13: {num_linhas}')

numeric_sums = df_ciclo_n13.select_dtypes(include=['number']).sum(numeric_only=True)
for coluna, soma in numeric_sums.items():
    print(f'{coluna}: {soma:.4f}')


# # # Tempos de Execução # # #

Ciclo: 18.00 segundos
ZP55: 3.49 segundos
ZP54: 7.25 segundos
GSV: 0.02 segundos
Projeção: 0.02 segundos
ZP53: 2.73 segundos
ZP52: 0.64 segundos
ZP73: 0.29 segundos
ZP70: 0.03 segundos
ZP39: 0.43 segundos
NIV: 0.02 segundos
Exportação: 357.91 segundos
Total: 394.43 segundos

Número de linhas do df_ciclo_n13: 351653
P03-2026: 17935.5335
P04-2026: 14898.9761
P05-2026: 16994.5423
P06-2026: 17152.2462
P07-2026: 17810.2952
P08-2026: 18376.3242
P09-2026: 18705.5491
P10-2026: 17374.9707
P11-2026: 17610.9109
P12-2026: 17367.7446
P13-2026: 10966.4056
kg/UN: 656044.2427
Ton/CDA: 2409.5347
Unid/CDA: 8304914.0000
LSV: 84231469.7600
CLIENTE: 884.2871
CD + UF DESTINO + Importação: 260.0815
CD + UF DESTINO + NCM: 80090.0749
CD + UF DESTINO + H05: 2115.6001
ZP55: 81945.0337
1. CLIENTE: -1451.7664
1. REDE: 336.4335
1. GP UF HIER 6: -14834.5780
1. GP UF HIER 5: -189778.5734
ZP54: -203569.1855
GSV/CDA: 42021537.6971
GSV/TON: 9647773177.6222
GSV R$ P03-2026 - 2026: 18660214